# Flow Matching Dissertation Experiments


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

## Generate Mixture Dataset

In [ ]:
def generate_mixture(n=5000):
    x1 = np.random.randn(n//2, 2) + [2,0]
    x2 = np.random.randn(n//2, 2) + [-2,0]
    return np.vstack([x1,x2])

X = generate_mixture()
plt.scatter(X[:,0], X[:,1], s=2)
plt.show()

## Gaussian Baseline

In [ ]:
mean = X.mean(0)
cov = np.cov(X.T)
def gaussian_nll(x):
    d = x-mean
    return 0.5*np.sum(d @ np.linalg.inv(cov) * d, axis=1)
print(gaussian_nll(X).mean())

## Flow Matching

In [ ]:
class Flow(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(2,128),torch.nn.ReLU(),torch.nn.Linear(128,2))
    def forward(self,x): return self.net(x)

model=Flow(); opt=torch.optim.Adam(model.parameters(),lr=1e-3)
X_t=torch.tensor(X,dtype=torch.float32)

for i in range(300):
    idx=torch.randint(0,len(X_t),(256,))
    b=X_t[idx]; n=torch.randn_like(b); t=torch.rand(len(b),1)
    xt=(1-t)*b+t*n; target=n-b
    loss=((model(xt)-target)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if i%100==0: print(i,loss.item())

## Sampling

In [ ]:
s=torch.randn(2000,2)
for _ in range(20): s=s+0.05*model(s)
s=s.detach().numpy()
plt.scatter(s[:,0],s[:,1],s=2); plt.show()

## Mode Coverage

In [ ]:
centers=np.array([[2,0],[-2,0]])
def covg(s):
    return np.mean([ (np.linalg.norm(s-c,axis=1)<1).mean() for c in centers ])
print(covg(s))